# ZaenDrive — Midterm Rental Assistant with Tools and RAG
**C115451101 — Zaen Alaabden Abdul Rahim**

Run the cells in order. A CPU runtime is enough. The model runs through your configured API.
Use your own course API key; never paste it into chat or a visible code cell.

The project is implemented, but the actual model responses and report screenshots must come from your own live run.


## 1. Upload the project ZIP
Select **C115451101_Zaen_Midterm_Project.zip**, not the teacher's original assignment ZIP.
This cell checks archive paths before extracting and starts in the correct project folder.


In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import os, sys

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one project ZIP.")
filename, archive_bytes = next(iter(uploaded.items()))
if not filename.lower().endswith(".zip"):
    raise ValueError("Choose the project ZIP file.")
extract_root = Path("/content/zaendrive_midterm")
extract_root.mkdir(exist_ok=True)
with ZipFile(BytesIO(archive_bytes)) as archive:
    for member in archive.infolist():
        target = (extract_root / member.filename).resolve()
        if not target.is_relative_to(extract_root.resolve()):
            raise ValueError("Unsafe ZIP path.")
    required = {'ZaenDrive/rental_tools.py', 'ZaenDrive/rag.py', 'ZaenDrive/scope_guard.py', 'ZaenDrive/restart_colab.py', 'ZaenDrive/documents/01_rental_handbook.md'}
    if not required.issubset(archive.namelist()):
        raise ValueError('Choose the NEW ZaenDrive Midterm project ZIP, not the old homework ZIP.')
    archive.extractall(extract_root)
PROJECT_DIR = extract_root / "ZaenDrive"
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print("Project ready:", PROJECT_DIR)
del uploaded, archive_bytes


## 2. Install packages
If Colab requests a session restart, restart it, rerun this cell, then continue with cell 3.
The path is set again here so the notebook also works after a session restart.


In [ ]:
from pathlib import Path
import os, sys, subprocess
PROJECT_DIR = Path("/content/zaendrive_midterm/ZaenDrive")
if not (PROJECT_DIR / "app.py").exists():
    raise FileNotFoundError("Run the ZIP upload cell first.")
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Dependencies installed. If Colab asks for a restart, restart and rerun this cell.")


## 3. Configure the API — secret input
The endpoint and model match your previous homework. This modified midterm version still needs a new live test with your own key.
If you use another tool-capable provider, change only its URL and model here.

Option A: add a Colab secret named **NKUST_API_KEY** in the key-shaped sidebar and allow this notebook to read it.
Option B: enter the key in the hidden prompt. It is kept in the current runtime's environment, not written into the notebook or project.


In [ ]:
import os
from getpass import getpass

os.environ["LLM_API_URL"] = "https://www.iai.nkust.edu.tw/aihub/v1/"
os.environ["MODEL_NAME"] = "Furen-large"
os.environ["MAX_STEPS"] = "6"
os.environ["MIN_CALL_INTERVAL"] = "5"
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"

api_key = ""
try:
    from google.colab import userdata
    api_key = userdata.get("NKUST_API_KEY") or ""
except Exception:
    pass
if not api_key.strip():
    api_key = getpass("Enter your course API key (hidden): ")
if not api_key.strip():
    raise ValueError("No API key was entered. Obtain the correct key from your course setup.")
os.environ["API_KEY"] = api_key.strip()
del api_key
print("API key loaded without displaying it. Model:", os.environ["MODEL_NAME"])


## 4. Local code checks — no API requests
These 33 tests check Python, retrieval and tool integration using scripted model responses. **They are not the assignment's live tool-choice score.**


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "unittest", "-v", "test_offline", "test_rag", "test_scope_guard"], check=True)


## 5. Start the chat
Open the displayed link. First ask **What are your opening hours?** and check that the tool panel shows `get_rental_info`.
Next ask **If I return the car early, will unused days be refunded automatically?** Check the cited answer and the retrieved handbook passage on the right. Then try the four cases in `START_HERE_DE.md`, each in a new conversation.
The share link opens this demo while the Colab runtime is active.


In [ ]:
import runpy
# Reload all project modules and reuse the current API environment.
_run = runpy.run_path("restart_colab.py", run_name="__main__")
demo = _run["demo"]


## 6. App routing and refusal checks — optional
Run after ordinary chat questions succeed. This checks 19 app cases, each in a fresh conversation. Allowed requests use the model; recognized scope attacks can be refused locally with zero model calls. Fixed-refusal cases check the exact answer as well as the absence of tools. This is not a raw model-accuracy score. Other answers, citations and argument values still need manual inspection. Do not use the chat at the same time. Keep the original failures and label the new run `after-scope-fix`.


In [ ]:
import subprocess, sys
RUN_LABEL = "after-scope-fix"
subprocess.run([sys.executable, "tool_choice_test.py", "--label", RUN_LABEL], check=True)


## 7. Download test evidence before ending the session
Download the JSON/CSV results for your report. Save your app screenshots separately.
The code submission goes on a public GitHub repository; this evidence ZIP is for keeping your own results.


In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
results = Path("test_results")
evidence = sorted(results.glob("*.json")) + sorted(results.glob("*.csv"))
if not evidence:
    raise FileNotFoundError("No test evidence exists yet. Run the live test first.")
archive_path = Path("/content/ZaenDrive_Midterm_test_results.zip")
with ZipFile(archive_path, "w", ZIP_DEFLATED) as archive:
    for path in evidence:
        archive.write(path, arcname=path.name)
files.download(str(archive_path))


## Before handing in
Use the instructor's original Word template in `report/`. Add 2–4 real conversation screenshots, including an honest missing-data or out-of-scope case. Enter the public midterm GitHub URL, actual model, real AI usage and working time. A routing score and a video are not required by the midterm instructions.
Export as **C115451101_Zaen_Midterm.pdf**. This app does not save bookings; leave the business-data-write checkbox unticked and skip its saved-data proof.
Upload the source, both CSVs, `documents/`, requirements.txt, .env.example, .gitignore, README.md and real screenshots to public GitHub. The notebook/tests may be included.
Never upload a real .env, database, key or credential-containing notebook output.